# Data loading and preprocessing

## Dataset loading

In [8]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import random_split

# For type hinting, which is good practice
from torch.utils.data import Dataset
from typing import Tuple

def load_fashion_data() -> Tuple[Dataset, Dataset, Dataset]:
    """
    Loads the Fashion-MNIST dataset and splits the data into train, val, test.

    This function loads the Fashion-MNIST training and test sets from
    torchvision. It then splits the original training set into a new,
    smaller training set and a validation set (90/10 ratio).

    Returns:
        Tuple[Dataset, Dataset, Dataset]: A tuple containing the training,
        validation, and test datasets as torch.utils.data.Dataset objects.
    """
    # Use a basic transform to convert images to tensors.
    # More complex transforms will be applied later in the custom Dataset class.
    # converts a PIL Image or numpy.ndarray to tensor.
    transform = transforms.ToTensor()

    # 1. Load the full original training data
    full_train_dataset = datasets.FashionMNIST(
        root='./data',
        train=True,
        download=True,
        transform=transform
    )

    # 2. Load the test data
    test_dataset = datasets.FashionMNIST(
        root='./data',
        train=False,
        download=True,
        transform=transform
    )

    # 3. Calculate split sizes
    dataset_size = len(full_train_dataset)
    train_size = int(0.9 * dataset_size)
    val_size = dataset_size - train_size

    # 4. Split the full training dataset into training and validation subsets
    # Using a fixed generator seed for reproducibility of the split
    generator = torch.Generator().manual_seed(42)
    train_dataset, val_dataset = random_split(
        full_train_dataset,
        [train_size, val_size],
        generator=generator
    )

    return train_dataset, val_dataset, test_dataset

# # Example of how to call the function and verify the output
# if __name__ == '__main__':
#     train_data, val_data, test_data = load_fashion_data()

#     print(f"Fashion-MNIST Dataset Summary:")
#     print(f"  - Total Original Training Images: {len(train_data) + len(val_data)}")
#     print(f"  - Split Training Set Size:      {len(train_data)}")
#     print(f"  - Split Validation Set Size:    {len(val_data)}")
#     print(f"  - Test Set Size:                {len(test_data)}")

## Custom Dataset Class

In [10]:
import torch
from torch.utils.data import Dataset, Subset
from torchvision import transforms
from typing import Tuple, Any

def calculate_mean_std(dataset: Dataset) -> Tuple[Tuple[float, ...], Tuple[float, ...]]:
    """
    Calculate the mean and standard deviation of a dataset.
    
    Args:
        dataset: A PyTorch dataset containing images
        
    Returns:
        Tuple containing mean and std as tuples
    """
    # Stack all images into a single tensor
    # We'll iterate through the dataset and collect all pixel values
    all_pixels = []
    
    for img, _ in dataset:
        # img is already a tensor from ToTensor() transform
        all_pixels.append(img)
    
    # Stack all images: shape will be [N, C, H, W]
    all_pixels = torch.stack(all_pixels)
    
    # Calculate mean and std across all dimensions except the channel dimension
    # Compute per-channel statistics
    mean = all_pixels.mean(dim=[0, 2, 3])  # Average over batch, height, width
    std = all_pixels.std(dim=[0, 2, 3])    # Std dev over batch, height, width
    
    return tuple(mean.tolist()), tuple(std.tolist())

# # Load the training data to calculate statistics
# train_dataset, val_dataset, test_dataset = load_fashion_data()

# # Calculate mean and std from the training set
# print("Calculating mean and standard deviation from training data...")
# FASHION_MNIST_MEAN, FASHION_MNIST_STD = calculate_mean_std(train_dataset)

# print(f"Calculated Mean: {FASHION_MNIST_MEAN}")
# print(f"Calculated Std:  {FASHION_MNIST_STD}")

In [13]:
# Load the training data to calculate statistics
train_dataset, val_dataset, test_dataset = load_fashion_data()

# Calculate mean and std from the training set
print("Calculating mean and standard deviation from training data...")
FASHION_MNIST_MEAN, FASHION_MNIST_STD = calculate_mean_std(train_dataset)

print(f"Calculated Mean: {FASHION_MNIST_MEAN}")
print(f"Calculated Std:  {FASHION_MNIST_STD}")

Calculating mean and standard deviation from training data...
Calculated Mean: (0.2856265902519226,)
Calculated Std:  (0.3527947962284088,)
Calculated Mean: (0.2856265902519226,)
Calculated Std:  (0.3527947962284088,)


In [14]:
class FashionMNISTDataset(Dataset):
    """
    Custom Dataset wrapper for Fashion-MNIST that returns:
    - Normalized image tensor
    - Class label
    - Ink target (normalized pixel intensity)
    """
    
    def __init__(
        self, 
        base_dataset: Dataset,
        mean: Tuple[float, ...],
        std: Tuple[float, ...]
    ):
        """
        Initialize the custom Fashion-MNIST dataset.
        
        Args:
            base_dataset: The underlying Fashion-MNIST dataset (can be a Subset)
            mean: Mean values for normalization (per channel)
            std: Standard deviation values for normalization (per channel)
        """
        self.base_dataset = base_dataset
        self.mean = mean
        self.std = std
        
        # Create normalization transform
        self.normalize = transforms.Normalize(mean=mean, std=std)
        
    def __len__(self) -> int:
        """Return the total number of samples in the dataset."""
        return len(self.base_dataset)
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int, float]:
        """
        Get a single item from the dataset.
        
        Args:
            idx: Index of the item to retrieve
            
        Returns:
            Tuple containing:
                - image: Normalized image tensor of shape [C, H, W]
                - class_label: Integer label in range [0, 9]
                - ink_target: Normalized pixel intensity (average pixel value)
        """
        # Get the original image and label from the base dataset
        image, class_label = self.base_dataset[idx]
        
        # Calculate ink target BEFORE normalization
        # Ink is the average pixel value (mean intensity) of the original image
        # image is already a tensor from ToTensor() with values in [0, 1]
        ink_target = image.mean().item()
        
        # Normalize the image using the dataset mean and std
        normalized_image = self.normalize(image)
        
        return normalized_image, class_label, ink_target

In [15]:
# Create custom dataset instances
train_custom = FashionMNISTDataset(train_dataset, FASHION_MNIST_MEAN, FASHION_MNIST_STD)
val_custom = FashionMNISTDataset(val_dataset, FASHION_MNIST_MEAN, FASHION_MNIST_STD)
test_custom = FashionMNISTDataset(test_dataset, FASHION_MNIST_MEAN, FASHION_MNIST_STD)

print("Custom datasets created successfully!")
print(f"Training set size: {len(train_custom)}")
print(f"Validation set size: {len(val_custom)}")
print(f"Test set size: {len(test_custom)}")

# Test the dataset by getting one sample
sample_image, sample_label, sample_ink = train_custom[0]
print(f"\nSample from dataset:")
print(f"  Image shape: {sample_image.shape}")
print(f"  Image dtype: {sample_image.dtype}")
print(f"  Class label: {sample_label}")
print(f"  Ink target: {sample_ink:.4f}")
print(f"  Image min value: {sample_image.min():.4f}")
print(f"  Image max value: {sample_image.max():.4f}")

Custom datasets created successfully!
Training set size: 54000
Validation set size: 6000
Test set size: 10000

Sample from dataset:
  Image shape: torch.Size([1, 28, 28])
  Image dtype: torch.float32
  Class label: 9
  Ink target: 0.3143
  Image min value: -0.8096
  Image max value: 2.0249
